In [3]:
import os
import duckdb
from dotenv import load_dotenv

load_dotenv('../.env')

endpoint = os.environ.get("S3_ENDPOINT", "localhost:8333")
access_key = os.environ["S3_ACCESS_KEY"]
secret_key = os.environ["S3_SECRET_KEY"]

con = duckdb.connect()

con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE OR REPLACE SECRET seaweedfs (
        TYPE s3,
        KEY_ID '{access_key}',
        SECRET '{secret_key}',
        ENDPOINT '{endpoint}',
        URL_STYLE 'path',
        USE_SSL false,
        REGION 'us-east-1'
    );
""")

RAW_PATH = "s3://raw/owid_energy/ingested_on=2026-10-06/owid-energy-data.csv"

df_raw = con.execute(f"SELECT * FROM read_csv_auto('{RAW_PATH}') LIMIT 5").fetchall()
for row in df_raw:
    print(row)

('ASEAN (Ember)', 2000, None, None, None, None, None, None, None, None, 5.87, 1.55, None, 572.58, None, None, None, None, None, 76.03, None, None, None, None, 20.073, None, 378.76, None, 378.76, None, None, None, None, None, None, None, None, 305.53, None, None, 80.666, None, None, None, None, None, 164.26, None, None, None, None, None, 43.368, None, 216.87, None, None, None, None, 50.43, None, 13.314, None, None, None, None, None, 73.23, None, 19.334, None, None, None, None, None, None, None, 0.0, None, 0.0, None, None, None, None, None, 65.24, None, None, None, None, None, 17.225, None, None, 22.8, 16.93, None, None, None, None, None, 6.02, 4.47, None, None, None, None, None, None, None, 73.23, None, 19.334, None, None, None, None, None, 0.0, None, 0.0, None, None, None, None, None, 0.0, None, 0.0, None, datetime.date(2026, 10, 6))
('ASEAN (Ember)', 2001, None, None, None, None, None, None, None, None, 6.46, 1.595, None, 570.27, None, None, None, None, None, 86.26, None, None, None, 

In [4]:
schema_info = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{RAW_PATH}')").fetchall()
total_columns = len(schema_info)
total_rows = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{RAW_PATH}')").fetchone()[0]
print(f"Всего строк: {total_rows}")
print(f"Всего колонок: {total_columns}")
nulls_query = f"""
SELECT 
    COUNT(*) AS total,
    COUNT(*) - COUNT(country) AS null_country,
    COUNT(*) - COUNT(year) AS null_year,
    COUNT(*) - COUNT(primary_energy_consumption) AS null_energy
FROM read_csv_auto('{RAW_PATH}')
"""
nulls_res = con.execute(nulls_query).fetchone()
print(f"Пропуски country: {nulls_res[1]}")
print(f"Пропуски year: {nulls_res[2]}")
print(f"Пропуски primary_energy_consumption: {nulls_res[3]}")
dupes_query = f"""
SELECT COUNT(*) FROM (
    SELECT country, year, COUNT(*) as cnt
    FROM read_csv_auto('{RAW_PATH}')
    GROUP BY country, year
    HAVING COUNT(*) > 1
)
"""
dupes_res = con.execute(dupes_query).fetchone()[0]
print(f"Дубликатов по (country, year): {dupes_res}")
period_query = f"SELECT MIN(year), MAX(year) FROM read_csv_auto('{RAW_PATH}')"
min_year, max_year = con.execute(period_query).fetchone()
print(f"Период: с {min_year} по {max_year} год")

Всего строк: 23377
Всего колонок: 131
Пропуски country: 0
Пропуски year: 0
Пропуски primary_energy_consumption: 10122
Дубликатов по (country, year): 0
Период: с 1900 по 2025 год


In [5]:
import pandas as pd
rules_query = f"""
WITH base AS (
    SELECT * FROM read_csv_auto('{RAW_PATH}')
)
SELECT 
    'rule_1_unique_key' as rule_id,
    COUNT(*) as checked_rows,
    (SELECT COUNT(*) FROM (SELECT country, year FROM base GROUP BY country, year HAVING COUNT(*)>1)) as violations,
    'critical' as severity,
    'quarantine' as action
FROM base
UNION ALL
SELECT 
    'rule_2_year_range',
    COUNT(*),
    COUNT(*) FILTER (WHERE year < 1900 OR year > 2025),
    'critical',
    'quarantine'
FROM base
UNION ALL
SELECT 
    'rule_3_energy_positive',
    COUNT(*) FILTER (WHERE primary_energy_consumption IS NOT NULL),
    COUNT(*) FILTER (WHERE primary_energy_consumption < 0),
    'critical',
    'quarantine'
FROM base
UNION ALL
SELECT 
    'rule_4_consistency_per_capita',
    COUNT(*) FILTER (WHERE energy_per_capita IS NOT NULL),
    COUNT(*) FILTER (WHERE energy_per_capita IS NOT NULL AND primary_energy_consumption IS NULL),
    'warning',
    'flag'
FROM base
UNION ALL
SELECT 
    'rule_5_country_not_empty',
    COUNT(*),
    COUNT(*) FILTER (WHERE TRIM(country) = '' OR country IS NULL),
    'critical',
    'quarantine'
FROM base
"""

df_rules = con.execute(rules_query).fetchdf()
display(df_rules)

ModuleNotFoundError: No module named 'pandas'

In [6]:
rules_query = f"""
WITH base AS (
    SELECT * FROM read_csv_auto('{RAW_PATH}')
)
SELECT 
    'rule_1_unique_key' as rule_id,
    COUNT(*) as checked_rows,
    (SELECT COUNT(*) FROM (SELECT country, year FROM base GROUP BY country, year HAVING COUNT(*)>1)) as violations,
    'critical' as severity,
    'quarantine' as action
FROM base
UNION ALL
SELECT 
    'rule_2_year_range',
    COUNT(*),
    COUNT(*) FILTER (WHERE year < 1900 OR year > 2025),
    'critical',
    'quarantine'
FROM base
UNION ALL
SELECT 
    'rule_3_energy_positive',
    COUNT(*) FILTER (WHERE primary_energy_consumption IS NOT NULL),
    COUNT(*) FILTER (WHERE primary_energy_consumption < 0),
    'critical',
    'quarantine'
FROM base
UNION ALL
SELECT 
    'rule_4_consistency_per_capita',
    COUNT(*) FILTER (WHERE energy_per_capita IS NOT NULL),
    COUNT(*) FILTER (WHERE energy_per_capita IS NOT NULL AND primary_energy_consumption IS NULL),
    'warning',
    'flag'
FROM base
UNION ALL
SELECT 
    'rule_5_country_not_empty',
    COUNT(*),
    COUNT(*) FILTER (WHERE TRIM(country) = '' OR country IS NULL),
    'critical',
    'quarantine'
FROM base
"""
res = con.execute(rules_query).fetchall()

print(f"{'rule_id':<30} | {'checked':<10} | {'violations':<10} | {'severity':<10} | {'action'}")
print("-" * 80)
for row in res:
    print(f"{row[0]:<30} | {row[1]:<10} | {row[2]:<10} | {row[3]:<10} | {row[4]}")

rule_id                        | checked    | violations | severity   | action
--------------------------------------------------------------------------------
rule_1_unique_key              | 23377      | 0          | critical   | quarantine
rule_2_year_range              | 23377      | 0          | critical   | quarantine
rule_3_energy_positive         | 13255      | 0          | critical   | quarantine
rule_4_consistency_per_capita  | 11086      | 0          | warning    | flag
rule_5_country_not_empty       | 23377      | 0          | critical   | quarantine


In [7]:
STAGING_PATH = "s3://staging/owid_energy/source_ingested_on=2026-10-06/staging.parquet"
QUARANTINE_PATH = "s3://staging/owid_energy/source_ingested_on=2026-10-06/quarantine.parquet"

staging_query = f"""
COPY (
    SELECT 
        *,
        CASE 
            WHEN energy_per_capita IS NOT NULL AND primary_energy_consumption IS NULL THEN TRUE 
            ELSE FALSE 
        END AS flag_inconsistent_capita
    FROM read_csv_auto('{RAW_PATH}')
    WHERE 
        NOT (year < 1900 OR year > 2025)
        AND NOT (primary_energy_consumption < 0)
        AND NOT (TRIM(country) = '' OR country IS NULL)
        AND (country, year) IN (
            SELECT country, year FROM read_csv_auto('{RAW_PATH}') GROUP BY country, year HAVING COUNT(*) = 1
        )
) TO '{STAGING_PATH}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE);
"""
con.execute(staging_query)
print("Файл staging.parquet успешно сохранен в S3!")

quarantine_query = f"""
COPY (
    SELECT 
        *,
        'rule_2_year_range, rule_3_energy_positive, rule_5_country_not_empty' AS quality_rule_ids,
        'Нарушение базовых ограничений (год вне диапазона, отрицательная энергия, пустая страна)' AS quarantine_reason
    FROM read_csv_auto('{RAW_PATH}')
    WHERE 
        (year < 1900 OR year > 2025)
        OR (primary_energy_consumption < 0)
        OR (TRIM(country) = '' OR country IS NULL)
) TO '{QUARANTINE_PATH}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE);
"""
con.execute(quarantine_query)
print("Файл quarantine.parquet успешно сохранен в S3!")

BinderException: Binder Error: Subquery returns 2 columns - expected 1
LINE 14:         AND (country, year) IN (
                                     ^

In [8]:
STAGING_PATH = "s3://staging/owid_energy/source_ingested_on=2026-10-06/staging.parquet"
QUARANTINE_PATH = "s3://staging/owid_energy/source_ingested_on=2026-10-06/quarantine.parquet"

staging_query = f"""
COPY (
    SELECT 
        *,
        CASE 
            WHEN energy_per_capita IS NOT NULL AND primary_energy_consumption IS NULL THEN TRUE 
            ELSE FALSE 
        END AS flag_inconsistent_capita
    FROM read_csv_auto('{RAW_PATH}')
    WHERE 
        NOT (year < 1900 OR year > 2025)
        AND NOT (primary_energy_consumption < 0)
        AND NOT (TRIM(country) = '' OR country IS NULL)
) TO '{STAGING_PATH}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE);
"""
con.execute(staging_query)
print("Файл staging.parquet успешно сохранен в S3!")

quarantine_query = f"""
COPY (
    SELECT 
        *,
        'rule_2_year_range, rule_3_energy_positive, rule_5_country_not_empty' AS quality_rule_ids,
        'Нарушение базовых ограничений (год вне диапазона, отрицательная энергия, пустая страна)' AS quarantine_reason
    FROM read_csv_auto('{RAW_PATH}')
    WHERE 
        (year < 1900 OR year > 2025)
        OR (primary_energy_consumption < 0)
        OR (TRIM(country) = '' OR country IS NULL)
) TO '{QUARANTINE_PATH}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE);
"""
con.execute(quarantine_query)
print("Файл quarantine.parquet успешно сохранен в S3!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Файл staging.parquet успешно сохранен в S3!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Файл quarantine.parquet успешно сохранен в S3!


In [9]:
staging_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{STAGING_PATH}')").fetchone()[0]
quarantine_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{QUARANTINE_PATH}')").fetchone()[0]
raw_count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{RAW_PATH}')").fetchone()[0]

print(f"Строк в raw: {raw_count}")
print(f"Строк в staging: {staging_count}")
print(f"Строк в карантине: {quarantine_count}")
print(f"Сумма (staging + карантин): {staging_count + quarantine_count}")
print("-" * 30)

if raw_count == (staging_count + quarantine_count):
    print("БАЛАНС СОШЕЛСЯ: raw = staging + карантин")
else:
    print("ОШИБКА: баланс не сошелся!")

critical_violations = con.execute(f"""
    SELECT COUNT(*) 
    FROM read_parquet('{STAGING_PATH}')
    WHERE (year < 1900 OR year > 2025) 
       OR (primary_energy_consumption < 0)
""").fetchone()[0]

print(f"Критических нарушений в staging: {critical_violations}")

flagged_rows = con.execute(f"""
    SELECT COUNT(*) 
    FROM read_parquet('{STAGING_PATH}')
    WHERE flag_inconsistent_capita = TRUE
""").fetchone()[0]

print(f"Записей с флагом flag_inconsistent_capita: {flagged_rows}")

print("\nПример данных из staging (первые 2 строки):")
res = con.execute(f"SELECT country, year, primary_energy_consumption, flag_inconsistent_capita FROM read_parquet('{STAGING_PATH}') LIMIT 2").fetchall()
for row in res:
    print(row)

Строк в raw: 23377
Строк в staging: 13255
Строк в карантине: 0
Сумма (staging + карантин): 13255
------------------------------
ОШИБКА: баланс не сошелся!
Критических нарушений в staging: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Записей с флагом flag_inconsistent_capita: 0

Пример данных из staging (первые 2 строки):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

('Taiwan', 1965, 70.358, False)
('Taiwan', 1966, 75.376, False)


In [10]:
staging_query = f"""
COPY (
    SELECT 
        *,
        CASE 
            WHEN energy_per_capita IS NOT NULL AND primary_energy_consumption IS NULL THEN TRUE 
            ELSE FALSE 
        END AS flag_inconsistent_capita
    FROM read_csv_auto('{RAW_PATH}')
    WHERE 
        -- Пропускаем, если год в порядке ИЛИ пустой
        (year BETWEEN 1900 AND 2025 OR year IS NULL)
        -- Пропускаем, если энергия >= 0 ИЛИ пустая
        AND (primary_energy_consumption >= 0 OR primary_energy_consumption IS NULL)
        -- Пропускаем, если страна не пустая
        AND (TRIM(country) != '' AND country IS NOT NULL)
) TO '{STAGING_PATH}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE);
"""
con.execute(staging_query)
print("Файл staging.parquet ПЕРЕЗАПИСАН с исправлениями!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Файл staging.parquet ПЕРЕЗАПИСАН с исправлениями!


In [11]:
staging_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{STAGING_PATH}')").fetchone()[0]
quarantine_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{QUARANTINE_PATH}')").fetchone()[0]
raw_count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{RAW_PATH}')").fetchone()[0]

print(f"Строк в raw: {raw_count}")
print(f"Строк в staging: {staging_count}")
print(f"Строк в карантине: {quarantine_count}")
print(f"Сумма (staging + карантин): {staging_count + quarantine_count}")
print("-" * 30)

if raw_count == (staging_count + quarantine_count):
    print("БАЛАНС СОШЕЛСЯ: raw = staging + карантин")
else:
    print("ОШИБКА: баланс не сошелся!")

critical_violations = con.execute(f"""
    SELECT COUNT(*) 
    FROM read_parquet('{STAGING_PATH}')
    WHERE (year < 1900 OR year > 2025) 
       OR (primary_energy_consumption < 0)
""").fetchone()[0]

print(f"Критических нарушений в staging: {critical_violations}")

flagged_rows = con.execute(f"""
    SELECT COUNT(*) 
    FROM read_parquet('{STAGING_PATH}')
    WHERE flag_inconsistent_capita = TRUE
""").fetchone()[0]

print(f"Записей с флагом flag_inconsistent_capita: {flagged_rows}")

print("\nПример данных из staging (первые 2 строки):")
res = con.execute(f"SELECT country, year, primary_energy_consumption, flag_inconsistent_capita FROM read_parquet('{STAGING_PATH}') LIMIT 2").fetchall()
for row in res:
    print(row)

Строк в raw: 23377
Строк в staging: 23377
Строк в карантине: 0
Сумма (staging + карантин): 23377
------------------------------
БАЛАНС СОШЕЛСЯ: raw = staging + карантин
Критических нарушений в staging: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Записей с флагом flag_inconsistent_capita: 0

Пример данных из staging (первые 2 строки):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

('Taiwan', 1936, None, False)
('Taiwan', 1937, None, False)
